# Hotel Booking with Priority Member Middleware

This notebook demonstrates **function-based middleware** using the Microsoft Agent Framework. We build upon the conditional workflow example by adding a middleware layer that gives priority members special privileges.

## What You'll Learn:
1. **Function-Based Middleware**: Intercept and modify function results
2. **Context Access**: Read and modify `context.result` after execution
3. **Business Logic Implementation**: Priority member benefits
4. **Result Override**: Change function outcomes based on user status
5. **Same Workflow, Different Outcomes**: Middleware-driven behavior changes

## Workflow Architecture with Middleware:

```
User Input: "I want to book a hotel in Paris"
                    ↓
        [availability_agent]
        - Calls hotel_booking tool
        - 🌟 priority_check middleware intercepts
        - Checks user membership status
        - IF priority + no rooms → Override to available!
        - Returns BookingCheckResult
                    ↓
        Conditional Routing
           /                    \
    [has_availability]    [no_availability]
          ↓                      ↓
    [booking_agent]        [alternative_agent]
    (Priority override!)   (Regular users)
          ↓                      ↓
       [display_result executor]
```

## Key Difference from Conditional Workflow:

**Without Middleware** (14-conditional-workflow.ipynb):
- Paris has no rooms → Route to alternative_agent

**With Middleware** (this notebook):
- Regular user + Paris → No rooms → Route to alternative_agent
- Priority user + Paris → 🌟 Middleware overrides! → Available → Route to booking_agent

## Prerequisites:
- Microsoft Agent Framework installed
- Understanding of conditional workflows (see 14-conditional-workflow.ipynb)
- GitHub token or OpenAI API key
- Basic understanding of middleware patterns


In [ ]:
import asyncio
import json
import os
from collections.abc import Awaitable, Callable
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    FunctionInvocationContext,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    executor,
    tool,
)
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")


## ಹಂತ 1: ರಚನೆಗೊಂಡ输出 ಗಾಗಿ ಪಿಡಾಂಟಿಕ್ ಮಾದರಿಗಳನ್ನು ವ್ಯಾಖ್ಯಾನಿಸಿ

ಈ ಮಾದರಿಗಳು ಏಜೆಂಟುಗಳು ಹಿಂತಿರುಗಿಸುವ **ಸ್ಕೀಮಾ** ಅನ್ನು ವ್ಯಾಖ್ಯಾನಿಸುತ್ತವೆ. ಮಧ್ಯವರ್ತಿ ಲಭ್ಯತೆ ಫಲಿತಾಂಶವನ್ನು ತಿದ್ದುಪಡಿ ಮಾಡಿದಾಗ ಟ್ರ್ಯಾಕ್ ಮಾಡಲು ನಾವು `priority_override` ಕ್ಷೇತ್ರವನ್ನು ಸೇರಿಸಿದ್ದೇವೆ.


In [ ]:
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""

    destination: str
    has_availability: bool
    message: str
    # Tracks if middleware overrode the result. The Azure structured-output
    # contract requires every property to be in the JSON schema's `required`
    # array, so we cannot give this a default value the way the original
    # notebook did.
    priority_override: bool


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""

    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""

    destination: str
    action: str
    message: str


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check with priority_override)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")

## ಹಂತ 2: ಪ್ರಾಥಮಿಕ ಸದಸ್ಯರ ಡೇಟಾಬೇಸ್ ಅನ್ನು ವ್ಯಾಖ್ಯಾನಿಸಿ

ಈ ಡೆಮೋಗೆ, ನಾವು ಪ್ರಾಥಮಿಕ ಸದಸ್ಯತ್ವ ಡೇಟಾಬೇಸ್ ಅನ್ನು ಅನುಕರಿಸೋಣ. ಉತ್ಪಾದನೆಯಲ್ಲಿ, ಇದು ನಿಜವಾದ ಡೇಟಾಬೇಸ್ ಅಥವಾ API ಅನ್ನು ಪ್ರಶ್ನಿಸುವುದು.

**ಪ್ರಾಥಮಿಕ ಸದಸ್ಯರು:**
- `alice@example.com` - VIP ಸದಸ್ಯ
- `bob@example.com` - ಪ್ರೀಮಿಯಮ್ ಸದಸ್ಯ  
- `priority_user` - ಪರೀಕ್ಷಾ ಖಾತೆ


In [ ]:
# Simulated priority members database
PRIORITY_MEMBERS = {
    "alice@example.com",
    "bob@example.com",
    "priority_user",
}

# Global variable to track current user (in real app, use proper session management)
current_user_id = "regular_user"  # Default: regular user


def set_user(user_id: str):
    """Set the current user for the session."""
    global current_user_id
    current_user_id = user_id
    is_priority = user_id in PRIORITY_MEMBERS
    status = "🌟 PRIORITY MEMBER" if is_priority else "👤 Regular User"

    display(
        HTML(f"""
        <div style='padding: 15px; background: {"linear-gradient(135deg, #FFD700 0%, #FFA500 100%)" if is_priority else "#e3f2fd"}; 
                    border-left: 4px solid {"#FF6B35" if is_priority else "#2196f3"}; border-radius: 4px; margin: 10px 0;'>
            <strong>👤 Current User Set:</strong> {user_id}<br>
            <strong>Status:</strong> {status}
        </div>
    """)
    )


print("✅ Priority members database created")
print(f"   Priority members: {len(PRIORITY_MEMBERS)} users")

## ಹಂತ 3: ಹೊಟೇಲ್ ಬುಕ್ಕಿಂಗ್ ಟೂಲ್ ಸೃಷ್ಟಿಸುವುದು

ಷರತುಪೂರ್ವಕ ಕಾರ್ಯಪ್ರವಾಹದಂತೆ, ಆದರೆ ಈಗ ಅದನ್ನು ಮಧ್ಯವರ್ತಿ ಮೂಲಕ ಅಡೆತಡೆಗೊಳ್ಳುತ್ತದೆ!


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.

    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

## ಹಂತ 4: 🌟 ಪ್ರಾಥಮಿಕತೆ ಪರೀಕ್ಷೆ ಮಧ್ಯವರ್ತಿ ಸೃಷ್ಟಿಸಿ (ಮುಖ್ಯ ವೈಶಿಷ್ಟ್ಯ!)

ಇದು ಈ ನೋಟ್‌ಬುಕ್‌ನ **ಮೂಲ ಕಾರ್ಯಾಚರಣೆ**. ಮಧ್ಯವರ್ತಿ:

1. ಹೋಟೆಲ್_ಬುಕಿಂಗ್ ಫಂಕ್ಷನ್ ಕರೆವನ್ನು **ನಿರೋಧಿಸುತ್ತದೆ**
2. `next(context)` ಅನ್ನು ಕರೆದು ಫಂಕ್ಷನ್ ಅನ್ನು ಸಾಮಾನ್ಯವಾಗಿ **ನಿರ್ವಹಿಸುತ್ತದೆ**
3. `context.result` ನಲ್ಲಿ ಫಲಿತಾಂಶವನ್ನು **ಸಮೀಕ್ಷಿಸುತ್ತದೆ**
4. ಬಳಕೆದಾರನು ಪ್ರಾಥಮಿಕತೆ ಹೊಂದಿರುವಾಗ ಮತ್ತು ಕೋಣೆಗಳಿಲ್ಲದಿದ್ದರೆ ಫಲಿತಾಂಶವನ್ನು **ತರಿಬದ್ಧಗೊಳಿಸುತ್ತದೆ**
5. ಪರಿವರ್ತಿತ ಫಲಿತಾಂಶವನ್ನು ಏಜೆಂಟ್‌ಗೆ **ಹಿಂತಿರುಗಿಸುತ್ತದೆ**

**ಮುಖ್ಯ ಮಾದರಿ:**
```python
async def my_middleware(context, next):
    await next(context)  # ಕಾರ್ಯವನ್ನು ನಿರ್ವಹಿಸಿ
    # ಈಗ context.result ನಲ್ಲಿ ಕಾರ್ಯದ ಔಟ್ಪುಟ್ ಇದೆ
    if some_condition:
        context.result = new_value  # ಮೀರಿಸುವಿಕೆ!
```


In [ ]:
async def priority_check_middleware(
    context: FunctionInvocationContext,
    next: Callable[[FunctionInvocationContext], Awaitable[None]],
) -> None:
    """
    Function middleware that overrides hotel_booking results for priority members.
    
    Workflow:
    1. Let the function execute normally
    2. Check if user is a priority member
    3. If priority + no availability → Override to make rooms available!
    4. Agent will then route to booking path instead of alternative path
    """
    function_name = context.function.name

    display(
        HTML(f"""
        <div style='padding: 12px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
            <strong>🔄 Middleware:</strong> Intercepting {function_name}...
        </div>
    """)
    )

    # Execute the original function
    await next(context)

    # Now inspect and potentially modify the result
    if context.result and function_name == "hotel_booking":
        result_data = json.loads(context.result)
        destination = result_data.get("destination", "")
        has_availability = result_data.get("has_availability", False)

        # Check if user is priority member
        is_priority = current_user_id in PRIORITY_MEMBERS

        # Override logic: Priority member + no availability → Make available!
        if is_priority and not has_availability:
            display(
                HTML(f"""
                <div style='padding: 20px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); 
                            border-radius: 8px; margin: 10px 0; box-shadow: 0 4px 12px rgba(255,165,0,0.4);'>
                    <h3 style='margin: 0 0 10px 0; color: #333;'>🌟 PRIORITY OVERRIDE ACTIVATED! 🌟</h3>
                    <p style='margin: 0; color: #555; line-height: 1.6;'>
                        <strong>User:</strong> {current_user_id}<br>
                        <strong>Status:</strong> VIP Priority Member<br>
                        <strong>Action:</strong> Overriding "No Availability" for {destination}<br>
                        <strong>Result:</strong> ✅ Rooms now available for priority booking!
                    </p>
                </div>
            """)
            )

            # Override the result!
            result_data["has_availability"] = True
            result_data["priority_override"] = True
            context.result = json.dumps(result_data)

        elif not has_availability:
            display(
                HTML(f"""
                <div style='padding: 12px; background: #ffebee; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                    <strong>ℹ️ Middleware:</strong> No priority override (user: {current_user_id})
                </div>
            """)
            )


print("✅ priority_check_middleware created")
print("   - Intercepts hotel_booking function")
print("   - Overrides availability for priority members")

## ಹೆಜ್ಜೆ 5: ಮಾರ್ಗಸೂಚಿಗಾಗಿ ಸ್ಥಿತಿ ಕಾರ್ಯಗಳನ್ನು నిర్వಚಿಸಿ

ಶರತ್ತು ಕಾರ್ಯಗಳೇ ಆದ್ದರಿಂದ ಶರತುಗಳ ಕೆಲಸದ ನಡವಳಿ - ಅವು ರೂಪಿತ output ಅನ್ನು ಪರಿಶೀಲಿಸಿ ಮಾರ್ಗ ನಿಶ್ಚಯಿಸುತ್ತವೆ.


In [ ]:
def has_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels ARE available (including priority overrides!)."""
    if not isinstance(message, AgentExecutorResponse):
        return True

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        # Check if this was a priority override
        override_indicator = " 🌟" if result.priority_override else ""

        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}{override_indicator}
            </div>
        """)
        )

        return result.has_availability
    except Exception as e:
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>⚠️  Error:</strong> {str(e)}
            </div>
        """)
        )
        return False


def no_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels are NOT available."""
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )

        return not result.has_availability
    except Exception:
        return False


print("✅ Condition functions defined")

## ಹಂತ 6: ಕಸ್ಟಮ್ ಪ್ರದರ್ಶನ ಕಾರ್ಯನಿರ್ವಾಹಕವನ್ನು ರಚಿಸಿ

ಹಿಂದಿನಂತೆ ಅದೇ ಕಾರ್ಯನಿರ್ವಾಹಕ - ಅಂತಿಮ ವರ್ಕ್‌ಫ್ಲೋ ಔಟ್‌ಪುಟ್ ಅನ್ನು ಪ್ರದರ್ಶಿಸುತ್ತದೆ.


In [ ]:
@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """Display the final result as workflow output."""
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ display_result executor created")

## ಹಂತ 7: ಪರಿಸರ ಚರಗಳನ್ನು ಲೋಡ್ ಮಾಡಿ

LLM ಕ್ಲೈಂಟ್ (Microsoft Foundry / Azure OpenAI) ಅನ್ನು ಕಾನ್ಫಿಗರ್ ಮಾಡಿ.


In [ ]:
# Load environment variables
load_dotenv()

# Configure the Microsoft Foundry provider with keyless authentication
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)


## ಹಂತ 8: ಮಿಡಲ್‌ವೇರ್‌ನೊಂದಿಗೆ AI ಏಂಜೆಂಟ್‌ಗಳನ್ನು ರಚಿಸಿ

**ಮುಖ್ಯ ವ್ಯತ್ಯಾಸ:** availability_agent ಅನ್ನು ರಚಿಸುವಾಗ, ನಾವು `middleware` ಪ್ಯಾರಾಮೀಟರ್ ಅನ್ನು ನೀಡುತ್ತೇವೆ!

ಇದು priority_check_middleware ಅನ್ನು ಏಂಜೆಂಟ್‌ನ ಕಾರ್ಯಾಚರಣೆ ಪೈಪ್ಲೈನ್‌ಗೆ ಹೇಗೆ ಸೇರಿಸುತ್ತೇವೆ ಎಂಬುದು.


In [ ]:
# Agent 1: Check availability with tool + middleware
availability_agent = AgentExecutor(
    provider.as_agent(
        name="availability-agent",
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), message (string), "
            "and priority_override (bool, true if priority member got special access). "
            "The message should summarize the availability status and mention if priority override occurred."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
        middleware=[priority_check_middleware],  # 🌟 MIDDLEWARE INJECTION!
    ),
    id="availability_agent",
)

# Agent 2: Suggest alternative (when no rooms)
alternative_agent = AgentExecutor(
    provider.as_agent(
        name="alternative-agent",
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 3: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    provider.as_agent(
        name="booking-agent",
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "If priority_override is true in the input, mention that they received priority member access. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created 3 Agents:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - WITH priority_check_middleware 🌟</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
        </ul>
    </div>
""")
)


## ಹೆಜ್ಜೆ 9: ವರ್ಕ್‌ಫ್ಲೋವನ್ನು ನಿರ್ಮಿಸಿ

ಹಿಂದೆ ಇದ್ದೇನಂತೆಯೇ ವರ್ಕ್‌ಫ್ಲೋ ರಚನೆ - ಲಭ್ಯತೆಯ ಆಧಾರದ ಮೇಲೆ ಷರತ್ತುಬದ್ಧ ಮಾರ್ಗ ನಿಗದಿ.


In [ ]:
# Build the workflow with conditional routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    # NO AVAILABILITY PATH
    .add_edge(availability_agent, alternative_agent, condition=no_availability_condition)
    .add_edge(alternative_agent, display_result)
    # HAS AVAILABILITY PATH (can be triggered by middleware override!)
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Conditional Routing (Middleware-Aware):</strong><br>
            • If <strong>NO availability</strong> → alternative_agent → display_result<br>
            • If <strong>availability</strong> (or 🌟 <strong>priority override</strong>) → booking_agent → display_result
        </p>
    </div>
""")
)

## ಹಂತ 10: ಪರೀಕ್ಷಾ ಪ್ರಕರಣ 1 - ಪ್ಯಾರಿಸ್‌ನಿನಲ್ಲಿ ಸಾಮಾನ್ಯ ಬಳಕೆದಾರ (ಓವರ್‌ರೈಡ್ ಇಲ್ಲದೆ)

ಸಾಮಾನ್ಯ ಬಳಕೆದಾರ প್ಯಾರಿಸ್ ಬುಕ್ ಮಾಡಲು ಯತ್ನಿಸುತ್ತಾನೆ → ಕೋಣೆಗಳು ಲಭ್ಯವಿಲ್ಲ → alternative_agent ಗೆ ಮಾರ್ಗನಿರ್ದೇಶನ


In [ ]:
# Set as regular user
set_user("regular_user")

display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Regular User + Paris</h3>
        <p style='margin: 0;'><strong>Expected:</strong> No rooms → No middleware override → Alternative suggestion</p>
    </div>
""")
)

# Create request
request_regular = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Paris")], should_respond=True
)

# Run workflow
events_regular = await workflow.run(request_regular)
outputs_regular = events_regular.get_outputs()

# Display results
if outputs_regular:
    result_regular = AlternativeResult.model_validate_json(outputs_regular[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: #fff; border: 2px solid #ff9800; border-radius: 12px; margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #e65100;'>📊 RESULT (Regular User)</h3>
            <div style='background: #fff3e0; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                <p style='margin: 0 0 10px 0;'><strong>Middleware:</strong> No priority override (regular user)</p>
                <p style='margin: 0 0 10px 0;'><strong>Alternative:</strong> 🏨 {result_regular.alternative_destination}</p>
                <p style='margin: 0;'><strong>Reason:</strong> {result_regular.reason}</p>
            </div>
        </div>
    """)
    )

## ಹಂತ 11: ಪರೀಕ್ಷಾ ಪ್ರಕರಣ 2 - 🌟 ಪ್ರಾಥಮಿಕ ಬಳಕೆದಾರ ಪ್ಯಾರಿಸ್ ನಲ್ಲಿ (ಒವರೈಡ್ ಸಹಿತ!)

ಒಂದು ಪ್ರಾಥಮಿಕ ಸದಸ್ಯನು ಪ್ಯಾರಿಸ್ ಅನ್ನು ಬುಕ್ ಮಾಡಲು ಪ್ರಯತ್ನಿಸುತ್ತಾನೆ → ಮೊದಲಿಗೆ ಯಾವುದೇ ಕೊಠಡಿಗಳು ಲಭ್ಯವಿಲ್ಲ → 🌟 ಮಿಡ್‌ಲ್‌ವೇರ್ ಒವರೈಡ್ ಮಾಡುತ್ತದೆ! → booking_agent ಗೆ ಮಾರ್ಗದರ್ಶನ

**ಇದು ಮಿಡ್‌ಲ್‌ವೇರ್ ಶಕ್ತಿಯ ಮುಖ್ಯ ಪ್ರದರ್ಶನ!**


In [ ]:
# Set as priority user
set_user("priority_user")

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #333;'>🧪 TEST CASE 2: 🌟 Priority User + Paris</h3>
        <p style='margin: 0; color: #555;'><strong>Expected:</strong> No rooms → 🌟 MIDDLEWARE OVERRIDE → Rooms available → Booking suggestion!</p>
    </div>
""")
)

# Create request
request_priority = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Paris")], should_respond=True
)

# Run workflow
events_priority = await workflow.run(request_priority)
outputs_priority = events_priority.get_outputs()

# Display results
if outputs_priority:
    result_priority = BookingConfirmation.model_validate_json(outputs_priority[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px;
                    box-shadow: 0 8px 16px rgba(255,165,0,0.4); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 RESULT (Priority Member) 🌟</h3>
            <div style='background: white; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available (Priority Override!)</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Middleware:</strong> 🌟 OVERRIDE ACTIVATED!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_priority.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_priority.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_priority.message}</p>
                <div style='margin-top: 15px; padding: 15px; background: #fff3cd; border-radius: 6px; border-left: 4px solid #FF6B35;'>
                    <strong>💡 What Just Happened:</strong><br>
                    1. hotel_booking tool returned "no availability"<br>
                    2. priority_check_middleware intercepted the result<br>
                    3. Middleware checked user status: priority_user ✅<br>
                    4. Middleware OVERRODE the result to "available"<br>
                    5. Workflow routed to booking_agent instead of alternative_agent!
                </div>
            </div>
        </div>
    """)
    )

## ಹಂತ 12: ಪರೀಕ್ಷಾ ಪ್ರಕರಣ 3 - ಸ್ಟೊಕ್‌ಹೋಲ್ಮ್‌ನಲ್ಲಿ ಪ್ರಾಥಮಿಕ ಬಳಕೆದಾರ (ಈಗಾಗಲೇ ಲಭ್ಯ)

ಪ್ರಾಥಮಿಕ ಬಳಕೆದಾರರು ಸ್ಟೊಕ್‌ಹೋಲ್ಮ್ ಪ್ರಯತ್ನಿಸುತ್ತಾರೆ → ಕೊಠಡಿಗಳು ಲಭ್ಯವಿವೆ → ಯಾವುದೇ ಮೀರಕ್ಷಣೆ ಅಗತ್ಯವಿಲ್ಲ → booking_agent ಗೆ ಮಾರ್ಗನಿರ್ದೇಶನ

ಇದು ಮಧ್ಯಮ ಸಾಫ್ಟ್‌ವೇರ್ ಅಗತ್ಯವಿರುವಾಗ ಮಾತ್ರ ಕ್ರಿಯಾಶೀಲವಾಗುತ್ತದೆ ಎಂದು ತೋರಿಸುತ್ತದೆ!


In [ ]:
# Priority user is still set from previous test

display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 3: Priority User + Stockholm</h3>
        <p style='margin: 0;'><strong>Expected:</strong> Rooms available → No override needed → Booking suggestion</p>
    </div>
""")
)

# Create request
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Stockholm")], should_respond=True
)

# Run workflow
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px;
                    box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 RESULT (Priority User - No Override Needed)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available (Natural)</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Middleware:</strong> No override needed</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
                <div style='margin-top: 15px; padding: 15px; background: #e8f5e9; border-radius: 6px; border-left: 4px solid #4caf50;'>
                    <strong>💡 Middleware Behavior:</strong><br>
                    • hotel_booking returned "available" naturally<br>
                    • Middleware saw available = true → No override needed<br>
                    • Workflow proceeded normally to booking_agent
                </div>
            </div>
        </div>
    """)
    )

## ಪ್ರಮುಖ ಟೇಕ್‌ಅವೆಸ್ ಮತ್ತು ಮಧ್ಯಸ್ಥಿಕೆಧಾರಕ ಪರಿಕಲ್ಪನೆಗಳು

### ✅ ನೀವು ಕಲಿತದ್ದು:

#### **1. ಕಾರ್ಯಾಧಾರಿತ ಮಧ್ಯಸ್ಥಿಕೆಧಾರಕ ಮಾದರಿ**

ಮಧ್ಯಸ್ಥಿಕೆಧಾರಕವು ಸರಳ ಅಸಿಂಕ್ರೋನ್ ಕಾರ್ಯವನ್ನು ಬಳಸಿ ಕಾರ್ಯಕಾಲಗಳನ್ನು ವಂಚಿಸುತ್ತದೆ:

```python
async def my_middleware(
    context: FunctionInvocationContext,
    next: Callable,
) -> None:
    # ಫಂಕ್ಷನ್ ಕಾರ್ಯಾಚರಣೆ ಮೊದಲು
    print("Intercepting...")
    
    # ಫಂಕ್ಷನ್ ಅನ್ನು ಕಾರ್ಯಗತಗೊಳಿಸಿ
    await next(context)
    
    # ಫಂಕ್ಷನ್ ಕಾರ್ಯಾಚರಣೆಯ ನಂತರ - ಫಲಿತಾಂಶ ಪರಿಶೀಲಿಸಿ
    if context.result:
        # ಅಗತ್ಯವಿದ್ದಲ್ಲಿ ಫಲಿತಾಂಶವನ್ನು ಪರಿವರ್ತಿಸಿ
        context.result = modified_value
```

#### **2. ಸੰਦਰಭ ಪ್ರವೇಶ ಮತ್ತು ಫಲಿತಾಂಶ ಮರುಬರಹ**

- `context.function` - ಕರೆ ಮಾಡಲಾಗುತ್ತಿರುವ ಕಾರ್ಯ ಪ್ರವೇಶಿಸಿ
- `context.arguments` - ಕಾರ್ಯದ ಸಂಚಿಕೆಗಳನ್ನು ಓದಿ
- `context.kwargs` - ಹೆಚ್ಚುವರಿ ಪ್ಯಾರಾಮೀಟರ್ಗಳನ್ನು ಪ್ರವೇಶಿಸಿ
- `await next(context)` - ಕಾರ್ಯವನ್ನು ನಿರ್ವಹಿಸಿ
- `context.result` - ಕಾರ್ಯದ ಔಟ್‌ಪುಟ್ ಅನ್ನು ಓದಿ/ಬದಲಾಯಿಸಿ

#### **3. ವ್ಯಾಪಾರ ಲಾಜಿಕ್ ಅನುಷ್ಠಾನ**

ನಮ್ಮ ಮಧ್ಯಸ್ಥಿಕೆಧಾರಕವು ಪ್ರಾಥಮಿಕ ಸದಸ್ಯರ ಲಾಭಗಳನ್ನು ಅನುಷ್ಠಾನಗೊಳಿಸುತ್ತದೆ:
- **ನಿಯಮಿತ ಬಳಕೆದಾರರು**: ಯಾವುದೇ ಬದಲಾವಣೆ ಇಲ್ಲ, ಮಾನದಂಡದ ವರ್ಕ್ಫ್ಲೋ
- **ಪ್ರಾಥಮಿಕ ಬಳಕೆದಾರರು**: "ಲಭ್ಯತೆ ಇಲ್ಲ" ಅನ್ನು "ಲಭ್ಯವಿದೆ" ಗೆ ಮರುಬರಹ ಮಾಡುತ್ತದೆ
- **ಶರತ್ಥಿತಿಗತ ತರ್ಕ**: ಅಗತ್ಯವಿದ್ದಾಗ ಮಾತ್ರ ಮರುಬರಹ ಮಾಡುತ್ತದೆ

#### **4. ಅದೇ ಕಾರ್ಯವಳಿ, ವಿವಿಧ ಫಲಿತಾಂಶಗಳು**

ಮಧ್ಯಸ್ಥಿಕೆಧಾರಕದ ಶಕ್ತಿ:
- ✅ ಕಾರ್ಯವಳಿ ರಚನೆಯಲ್ಲಿ ಯಾವುದೇ ಬದಲಾವಣೆ ಇಲ್ಲ
- ✅ ಉಪಕರಣ ಕಾರ್ಯದಲ್ಲಿ ಯಾವುದೇ ಬದಲಾವಣೆ ಇಲ್ಲ
- ✅ ಶರತ್ಥಿತಿಗತ ಮಾರ್ಗಸೂಚಿ ತರ್ಕದಲ್ಲಿ ಯಾವುದೇ ಬದಲಾವಣೆ ಇಲ್ಲ
- ✅ ಕೇವಲ ಮಧ್ಯಸ್ಥಿಕೆಧಾರಕ → ವಿಭಿನ್ನ ನಡತ!

### 🚀 ವಾಸ್ತವಿಕ ಜಗತ್ತಿನ ಅನ್ವಯಿಕೆಗಳು:

1. **ವಿಐಪಿ/ಪ್ರೀಮಿಯಂ ವೈಶಿಷ್ಟ್ಯಗಳು**
   - ಪ್ರೀಮಿಯಂ ಬಳಕೆದಾರರಿಗೆ ದರದ ಮಿತಿ ಮರುಬರಹ
   - ಸಂಪನ್ಮೂಲಗಳಿಗೆ ಪ್ರಾಥಮಿಕ ಪ್ರವೇಶ ಒದಗಿಸಿ
   - ಪ್ರೀಮಿಯಂ ವೈಶಿಷ್ಟ್ಯಗಳನ್ನು ಡೈನಾಮಿಕ್‌ ಆಗಿ ಅನ್ಲಾಕ್ ಮಾಡಿ

2. **ಎ/ಬಿ ಟೆಸ್ಟಿಂಗ್**
   - ಬಳಕೆದಾರರನ್ನು ವಿಭಿನ್ನ ಅನುಷ್ಠಾನಗಳಿಗೆ ಮಾರ್ಗಸೂಚಿ ನೀಡಿ
   - ನವೀನ ವೈಶಿಷ್ಟ್ಯಗಳನ್ನು ನಿರ್ದಿಷ್ಟ ಬಳಕೆದಾರರೊಂದಿಗೆ ಪರೀಕ್ಷಿಸಿ
   - ಹಂತ ಹಂತವಾಗಿ ವೈಶಿಷ್ಟ್ಯವನ್ನು ಬಿಡುಗಡೆ ಮಾಡಿ

3. **ಭದ್ರತೆ ಮತ್ತು ಅನುಕೂಲತೆ**
   - ಕಾರ್ಯಕಾಲಗಳನ್ನು ಪರಿಶೀಲಿಸಿ
   - ಸಂವೇದನಾಶೀಲ ಕಾರ್ಯಗಳನ್ನು ನಿರೋಧಿಸಿ
   - ವ್ಯಾಪಾರ ನಿಯಮಗಳನ್ನು ಜಾರಿಗೆ ತರಿರಿ

4. **ಕಾರ್ಯಕ್ಷಮತೆಯ ಸುಧಾರಣೆ**
   - ನಿರ್ದಿಷ್ಟ ಬಳಕೆದಾರರಗಾಗಿ ಫಲಿತಾಂಶಗಳನ್ನು ನಕಲಿಸಿ
   - ಸಾಧ್ಯವಾದರೆ ವಿಸರ್ಜನೀಯ ಕಾರ್ಯಗಳನ್ನು ತಪ್ಪಿಸಿರಿ
   - ಡೈನಾಮಿಕ್ ಸಂಪನ್ಮೂಲ ಹಂಚಿಕೆಯನ್ನು ಅನುಷ್ಠಾನಗೊಳಿಸಿ

5. **ದೋಷ ನಿರ್ವಹಣೆ ಮತ್ತು ಮರುಪ್ರಯತ್ನ**
   - ದೋಷಗಳನ್ನು ಹಿಡಿದು ಮತ್ತು ಸೌಮ್ಯವಾಗಿ ನಿರ್ವಹಿಸಿ
   - ಮರುಪ್ರಯತ್ನ ತರ್ಕವನ್ನು ಅನುಷ್ಠಾನಗೊಳಿಸಿ
   - ಪರ್ಯಾಯ ಅನುಷ್ಠಾನಗಳಿಗೆFallback ಮಾಡಿ

6. **ಲಾಗಿಂಗ್ ಮತ್ತು ನಿಗಾ ವಹಿಸಿ**
   - ಕಾರ್ಯ ಕಾರ್ಯನಿರ್ವಹಣಾ ಸಮಯವನ್ನು ಟ್ರ್ಯಾಕ್ ಮಾಡಿ
   - ಪ್ಯಾರಾಮೀಟರ್‌ಗಳು ಮತ್ತು ಫಲಿತಾಂಶಗಳನ್ನು ಲಾಗ್ ಮಾಡಿ
   - ಬಳಕೆ ಮಾದರಿಗಳನ್ನು ನಿಗಾ ವಹಿಸಿ

### 🔑 ಡೆಕೊರೇಟರ್‌ಗಳಿಂದ ಮಹತ್ವದ ವ್ಯತ್ಯಾಸಗಳು:

| ವೈಶಿಷ್ಟ್ಯ | ಡೆಕೊರೇಟರ್ | ಮಧ್ಯಸ್ಥಿಕೆಧಾರಕ |
|---------|-----------|------------|
| ** ವ್ಯಾಪ್ತಿ ** | ಏಕ ಕಾರ್ಯ | ಏಜೆಂಟ್‌ನ ಎಲ್ಲಾ ಕಾರ್ಯಗಳು |
| **ಬದಲಾವಣೆ ಸಾಮರ್ಥ್ಯ** | ವ್ಯಾಖ್ಯಾನದ ವೇಳೆ ನಿಶ್ಚಿತ | ಚಾಲನೆ ಸಮಯದಲ್ಲಿ ಡೈನಾಮಿಕ್ |
| **ಸಂದರ್ಬ** | ಸೀಮಿತ | ಪೂರ್ಣ ಏಜೆಂಟ್ ಸಂದರ್ಬ |
| **ಸಂಯೋಜನೆ** | ಅನೇಕ ಡೆಕೊರೇಟರ್‌ಗಳು | ಮಧ್ಯಸ್ಥಿಕೆಧಾರಕ ಪೈಪುಲೈನ್ |
| **ಏಜೆಂಟ್ ಜ್ಞಾತ** | ಇಲ್ಲ | ಹೌದು (ಏಜೆಂಟ್ ಸ್ಥಿತಿಗೆ ಪ್ರವೇಶ) |

### 📚 ಯಾವಾಗ ಮಧ್ಯಸ್ಥಿಕೆಧಾರಕವನ್ನು ಉಪಯೋಗಿಸಬೇಕು:

✅ **ಮಧ್ಯಸ್ಥಿಕೆಧಾರಕವನ್ನು ಉಪಯೋಗಿಸಿ, երբ:**
- ಬಳಕೆದಾರ/ಸೆಷನ್ ಸ್ಥಿತಿಯ ಆಧಾರದಲ್ಲಿ ನಡತೆಯನ್ನು ಬದಲಾಯಿಸಲು ಅಗತ್ಯವಿದ್ದಾಗ
- ಹಲವು ಕಾರ್ಯಗಳಿಗೆ ಲಾಜಿಕ್ ಅನ್ವಯಿಸಲು ಬಯಸಿದಾಗ
- ಏಜೆಂಟ್ ಮಟ್ಟದ ಸੰਦਰಭವನ್ನು ಪ್ರವೇಶಿಸಲು ಅಗತ್ಯವಿದ್ದಾಗ
- ನೀವು ಕ್ರಾಸ್-ಕಟಿಂಗ್ ವಿಚಾರಗಳನ್ನು (ಲಾಗಿಂಗ್, ಪ್ರಾಮಾಣೀಕರಣ ಮತ್ತಿತರ) ಅನುಷ್ಠಾನಗೊಳಿಸುತ್ತಿದ್ದಲ್ಲಿ

❌ **ಮಧ್ಯಸ್ಥಿಕೆಧಾರಕವನ್ನು ಬಳಸಬೇಡಿ, երբ:**
- ಸರಳ ಇನ್‌ಪುಟ್ ಮಾನ್ಯತೆ (ಪೈಡಾಂಟಿಕ್ ಉಪಯೋಗಿಸಿ)
- ಕಾರ್ಯ-ನಿರ್ದಿಷ್ಟ ತರ್ಕ (ಕಾರ್ಯದಲ್ಲಿರಲಿ)
- ಒಮ್ಮೆ ಬದಲಾವಣೆಗಳು (ಕೆವಲ ಕಾರ್ಯ ಬದಲಿಸಿ)

### 🎓 ಉತ್ತೀರ್ಣ ಮಾದರಿಗಳು:

```python
# ಹಲವು ಮಧ್ಯವರ್ತಿಗಳು (ನಿರ್ವಹಣೆಯ ಕ್ರಮ ಪ್ರಮುಖವಾಗಿದೆ!)
middleware=[
    logging_middleware,      # ಮೊದಲು ಲಾಗ್‌ಗಳು
    auth_middleware,         # ನಂತರ ಪ್ರಾಮಾಣಿಕತೆ ಪರಿಶೀಲನೆ
    cache_middleware,        # ನಂತರ ಕ್ಯಾಶೆ ಪರಿಶೀಲನೆ
    rate_limit_middleware,   # ನಂತರ ದರ ಮಿತಿ
    priority_check_middleware  # ಕೊನೆಗೆ ಪ್ರಾಮುಖ್ಯತೆ ಪರಿಶೀಲನೆ
]

# ಶರતી ಮಧ್ಯವರ್ತಿ ಕಾರ್ಯನಿರ್ವಹಣೆ
async def conditional_middleware(context, next):
    if should_execute(context):
        await next(context)
        # ಫಲಿತಾಂಶವನ್ನು ಪರಿವರ್ತಿಸಿ
    else:
        # ಕಾರ್ಯನಿರ್ವಹಣೆಯನ್ನು ಸಂಪೂರ್ಣವಾಗಿ ಬಿಟ್ಟುಬಿಡಿ
        context.result = cached_value
```

### 🔗 ಸಂಬಂಧಿತ ಪರಿಕಲ್ಪನೆಗಳು:

- **ಏಜೆಂಟ್ ಮಧ್ಯಸ್ಥಿಕೆಧಾರಕ**: agent.run() ಕರೆಗಳನ್ನು ಮಧ್ಯಸ್ಥಿಕೆ ಮಾಡುತ್ತದೆ
- **ಕಾರ್ಯ ಮಧ್ಯಸ್ಥಿಕೆಧಾರಕ**: ಉಪಕರಣ ಕಾರ್ಯಕಾಲಗಳನ್ನು ಮಧ್ಯಸ್ಥಿಕೆ ಮಾಡುತ್ತದೆ (ನಾವು ಬಳಸಿದದ್ದು!)
- **ಮಧ್ಯಸ್ಥಿಕೆಧಾರಕ ಪೈಪುಲೈನ್**: ಕ್ರಮವಾಗಿ ಕಾರ್ಯನಿರ್ವಹಿಸುವ ಮಧ್ಯಸ್ಥಿಕೆಧಾರಕದ ಸರಪಳಿ
- **ಸಂದರ್ಬ ಹರಿವು**: ಮಧ್ಯಸ್ಥಿಕೆಧಾರಕ ಸರಪಳಿಯ ಮೂಲಕ ಸ್ಥಿತಿಯನ್ನು ಪಾಸು ಮಾಡುತ್ತದೆ


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ಅಸ್ವೀಕಾರ**:
ಈ ದಸ್ತಾವೇಜು AI ಅನುವಾದ ಸೇವೆ [Co-op Translator](https://github.com/Azure/co-op-translator) ಬಳಸಿ ಅನುವಾದಿಸಲಾಗಿದೆ. ನಾವು ನಿಖರತೆಯನ್ನು ಸಾಧಿಸಲು ಪ್ರಯತ್ನಿಸುತ್ತಿದ್ದರೂ, ದಯವಿಟ್ಟು ಗಮನಿಸಿ, ಸ್ವಯಂಚಾಲಿತ ಅನುವಾದಗಳಲ್ಲಿ ದೋಷಗಳು ಅಥವಾ ಅಸಡ್ಡೆಗಳು ಇರಬಹುದು. ಮೂಲ ಭಾಷೆಯಲ್ಲಿರುವ ಮೂಲ ದಸ್ತಾವೇಜು ಪ್ರಾಮಾಣಿಕ ಮೂಲವೆಂದು ಪರಿಗಣಿಸಬೇಕು. ಪ್ರಮುಖ ಮಾಹಿತಿಗಾಗಿ, ವೃತ್ತಿಪರ ಮಾನವ ಅನುವಾದವನ್ನು ಶಿಫಾರಸು ಮಾಡಲಾಗುತ್ತದೆ. ಈ ಅನುವಾದವನ್ನು ಬಳಸುವ ಮೂಲಕ ಉಂಟಾಗುವ ಯಾವುದೇ ತಪ್ಪು ಅರ್ಥಗಳ ಅಥವಾ ತಪ್ಪು ವ್ಯಾಖ್ಯಾನಗಳ ಬಗ್ಗೆ ನಾವು ಹೊಣೆಗಾರರಲ್ಲ.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
